# Data Source 03 · 제조·품질

> **솔직한 한계:** 달력 시간이 있는 무료 제조 시계열은 드물다. 아래 "추가" 셀에서 작은 공개 자료(AI4I, Steel Plates, SECOM, C-MAPSS, CWRU)를 다룬다. 이들은 대부분 **달력 시계열이 아니라** 표·사이클·신호 자료다.

이 노트북의 기본 경로(아래 셀)는 **합성 공정 자료 + 관리도 시연**이다. 실데이터 로더는 "추가" 셀에 있다.

In [ ]:
import io, json, urllib.request, numpy as np, pandas as pd, matplotlib.pyplot as plt

def fetch_bytes(url, timeout=30):
    """무키 공개 URL 요청. 실패 시 예외 -> 호출부에서 합성 대체."""
    req = urllib.request.Request(url, headers={"User-Agent": "ts-lab-course/1.0 (education)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

def synthetic_series(n, freq, seed, trend=0.05, season=12, noise=1.0, start="2015-01-01"):
    """NOT real data: 방법 시연용 합성 시계열."""
    rng = np.random.default_rng(seed)
    t = np.arange(n)
    y = 100 + trend * t + 3 * np.sin(2 * np.pi * t / season) + rng.normal(0, noise, n)
    return pd.Series(y, index=pd.date_range(start, periods=n, freq=freq), name="value")

def quality_report(s):
    s = s.sort_index()
    print("관측 수:", len(s), "| 기간:", s.index.min(), "~", s.index.max())
    print("결측:", int(s.isna().sum()), "| 중복 시점:", int(s.index.duplicated().sum()))
    f = pd.infer_freq(s.index) if len(s) > 2 else None
    print("추정 주기:", f)

In [ ]:
def synthetic_process(n=400, seed=2026031, shift_at=250, shift=1.2):
    """NOT real data: 평균 이동이 있는 공정 측정값 (관리도 시연용)."""
    rng = np.random.default_rng(seed)
    x = rng.normal(10, 0.5, n)
    x[shift_at:] += shift
    return pd.Series(x, index=pd.RangeIndex(n), name="measure"), shift_at

In [ ]:
USE_SECOM = False   # True로 바꾸면 UCI SECOM을 시도 (pip install ucimlrepo 필요, 구조는 직접 확인)
DATA_KIND = None
if USE_SECOM:
    try:
        from ucimlrepo import fetch_ucirepo
        secom = fetch_ucirepo(id=179)
        X = secom.data.features
        print("SECOM 특성 shape:", X.shape, "| 타임스탬프 열 여부는 secom.data.original 확인")
        DATA_KIND = "REAL: UCI SECOM (구조 확인 필요)"
    except Exception as e:
        print("SECOM 수집 실패 -> 합성 대체:", type(e).__name__, e)
if DATA_KIND is None:
    y, brk = synthetic_process()
    DATA_KIND = "SYNTHETIC (NOT real data)"
    mu0, sd0 = y.iloc[:brk].mean(), y.iloc[:brk].std()
    fig, ax = plt.subplots()
    ax.plot(y.index.to_numpy(), y.to_numpy()); ax.set_title(DATA_KIND)
    ax.axhline(mu0 + 3 * sd0, ls="--"); ax.axhline(mu0 - 3 * sd0, ls="--"); plt.show()
print("DATA_KIND:", DATA_KIND)

## 점검 체크리스트
- 관측 단위가 **시간(달력)**인지, 로트·사이클 번호인지: 사이클 자료에 달력 기반 ARIMA/계절성을 적용하지 않는다
- 라벨(합불)의 시점 정렬과 누수 여부 (Ch.8)
- 센서 결측·드리프트와 교정 시점
- 라이선스 확인 전 원자료 재배포 금지

## 추가 · 작은 공개 제조 자료 (용량 순)

| 자료 | 용량·규모 | 라이선스 | 검증 | 성격 |
|---|---|---|---|---|
| UCI AI4I 2020 (id 601) | 약 0.5MB, 10,000행 | CC BY 4.0 | A (페이지 확인) | **합성 생성 표**. 행이 독립적이며 시계열 아님 |
| UCI Steel Plates Faults (id 198) | 약 0.1MB, 1,941×27 | CC BY 4.0 | A | 결함 유형 분류 표 |
| UCI SECOM (id 179) | 약 1.9MB, 1,567×591 | CC BY 4.0 | A | 공정 센서 + 합불. 결측 많음. 타임스탬프는 라벨 파일에 있는 것으로 알려져 있음(확인 필요) |
| NASA C-MAPSS FD001 | 텍스트 ZIP(용량 미확인) | 표기 없음 | B (카탈로그 페이지 확인) | **시뮬레이션** 엔진 열화(사이클 단위, 달력 시간 아님) |
| CWRU 베어링 `.mat` | 파일당 수 MB 추정 | 이용약관 표기 없음 | B | 진동 신호. `.mat` 직접 URL은 사이트에서 복사해 `CWRU_URLS`에 넣을 것(미검증) |

**주의:** AI4I의 고장 유형 열(TWF, HDF, PWF, OSF, RNF)은 `machine_failure`를 거의 결정하므로 특성으로 쓰면 **누수**다. 이 저장소 코드는 이 열들을 분리한다.
이 표들은 **달력 시계열이 아니므로** ARIMA·계절성 분석에 쓰지 않는다. C-MAPSS는 사이클 기반이고, CWRU는 고주파 신호 분류용이다.

In [ ]:
def parse_secom(data_text, labels_text):
    """secom.data(공백 구분, 'NaN' 결측) + secom_labels.data(라벨 [날짜 시간]). 타임스탬프 열이 있으면 index로 쓴다."""
    X = pd.read_csv(io.StringIO(data_text), sep=r"\s+", header=None, na_values=["NaN"])
    lab = pd.read_csv(io.StringIO(labels_text), sep=r"\s+", header=None)
    if len(lab) != len(X):
        raise ValueError("SECOM 행 수 불일치")
    y = pd.to_numeric(lab[0]).map({-1: 0, 1: 1})        # 1 = 불량(fail)
    if y.isna().any():
        raise ValueError("SECOM 라벨 값 이상(-1/1 외)")
    if lab.shape[1] >= 3:
        X.index = pd.to_datetime(lab[1] + " " + lab[2], dayfirst=True, errors="coerce")
    X.columns = [f"f{i}" for i in range(X.shape[1])]
    return X, y.rename("fail").set_axis(X.index)

def synthetic_secom(n=1567, p=60, seed=2026103):
    """NOT real data: 결측·불균형이 있는 합성 공정 센서 표."""
    rng = np.random.default_rng(seed)
    X = pd.DataFrame(rng.normal(0, 1, (n, p)), columns=[f"f{i}" for i in range(p)],
                     index=pd.date_range("2008-07-19", periods=n, freq="40min"))
    y = pd.Series((rng.random(n) < 0.066 + 0.03 * (X["f0"] > 1.5)).astype(int), index=X.index, name="fail")
    X.loc[y == 1, "f1"] += 0.8
    X = X.mask(rng.random((n, p)) < 0.05)
    return X, y

STEEL_CLASSES = ["Pastry", "Z_Scratch", "K_Scatch", "Stains", "Dirtiness", "Bumps", "Other_Faults"]

def parse_steel(df):
    d = df.copy()
    cols = [c for c in d.columns if str(c).strip() in STEEL_CLASSES]
    if len(cols) != 7:
        cols = list(d.columns[-7:])                      # 열 이름이 다르면 마지막 7열을 결함 지시열로 가정 (확인 필요)
    y = d[cols].astype(int).values.argmax(axis=1)
    X = d.drop(columns=cols).select_dtypes("number")
    if len(d) < 1500 or X.shape[1] < 20:
        raise ValueError("Steel Plates 길이/특성 수 이상")
    return X.reset_index(drop=True), pd.Series(y, name="fault_class")

def synthetic_steel(n=1941, p=27, seed=2026104):
    rng = np.random.default_rng(seed)
    y = rng.integers(0, 7, n); X = pd.DataFrame(rng.normal(0, 1, (n, p)), columns=[f"x{i}" for i in range(p)])
    for k in range(7):
        X.loc[y == k, f"x{k}"] += 1.5
    return X, pd.Series(y, name="fault_class")

def parse_cwru_mat(raw):
    """CWRU .mat -> 구동단(DE) 가속도 신호. 변수명 규칙: 'X###_DE_time'."""
    from scipy.io import loadmat
    m = loadmat(io.BytesIO(raw))
    key = [k for k in m if k.endswith("_DE_time")]
    if not key:
        raise ValueError("DE_time 변수 없음: " + ", ".join(k for k in m if not k.startswith("__")))
    return np.asarray(m[key[0]]).ravel().astype(float)

def synthetic_bearing(fs=12000, seconds=4, fault_hz=0.0, seed=2026105):
    """NOT real data: 공진 + 잡음, fault_hz>0이면 그 주기로 충격(외륜결함 유사). 주파수 값은 가정이다."""
    rng = np.random.default_rng(seed); n = fs * seconds; t = np.arange(n) / fs
    x = 0.05 * np.sin(2 * np.pi * 30 * t) + rng.normal(0, 0.05, n)
    if fault_hz > 0:
        for t0 in np.arange(0, seconds, 1 / fault_hz):
            i = int(t0 * fs); k = np.arange(min(200, n - i))
            x[i:i + len(k)] += 0.8 * np.exp(-k / 40) * np.sin(2 * np.pi * 3000 * k / fs)
    return x

def bearing_features(x):
    from scipy.stats import kurtosis
    return dict(rms=float(np.sqrt(np.mean(x ** 2))), kurtosis=float(kurtosis(x)), peak=float(np.max(np.abs(x))))

import zipfile
AI4I_COLS = ["type", "air_temperature", "process_temperature", "rotational_speed", "torque", "tool_wear", "machine_failure"]
AI4I_MODES = ["twf", "hdf", "pwf", "osf", "rnf"]            # 고장 유형 열: 'machine_failure'를 거의 결정 -> 특성으로 쓰면 누수

def _norm_cols(df):
    out = df.copy()
    out.columns = [pd.Series(str(c)).str.replace(r"\[.*?\]", "", regex=True).str.strip().str.lower().str.replace(r"\s+", "_", regex=True).iloc[0]
                   for c in out.columns]
    return out

def parse_ai4i(df):
    d = _norm_cols(df)
    missing = [c for c in AI4I_COLS if c not in d.columns]
    if missing:
        raise ValueError("AI4I 예상 열 없음: " + ", ".join(missing))
    d = d[AI4I_COLS + [c for c in AI4I_MODES if c in d.columns]].copy()
    d["machine_failure"] = pd.to_numeric(d["machine_failure"]).astype(int)
    if len(d) < 5000 or not d["machine_failure"].isin([0, 1]).all():
        raise ValueError("AI4I 길이/라벨 이상")
    return d.reset_index(drop=True)

def synthetic_ai4i(n=10000, seed=2026101):
    """NOT real data: AI4I와 같은 열 구성의 합성 표. 고장은 규칙으로 생성(실제 현상 아님)."""
    rng = np.random.default_rng(seed)
    typ = rng.choice(["L", "M", "H"], n, p=[0.6, 0.3, 0.1])
    air = rng.normal(300, 2, n); proc = air + 10 + rng.normal(0, 1, n)
    rpm = rng.normal(1540, 170, n).clip(1100, 2900); torque = rng.normal(40, 6, n).clip(3, 80)
    wear = rng.integers(0, 250, n).astype(float)
    hdf = ((proc - air) < 8.6) & (rpm < 1380)
    power = torque * rpm * 2 * np.pi / 60
    pwf = (power < 3500) | (power > 9000)
    osf = wear * torque > np.where(typ == "L", 11000, np.where(typ == "M", 12000, 13000))
    twf = (wear > 200) & (rng.random(n) < 0.02)
    rnf = rng.random(n) < 0.001
    fail = (hdf | pwf | osf | twf | rnf).astype(int)
    return pd.DataFrame(dict(type=typ, air_temperature=air, process_temperature=proc, rotational_speed=rpm, torque=torque,
                             tool_wear=wear, machine_failure=fail, twf=twf.astype(int), hdf=hdf.astype(int),
                             pwf=pwf.astype(int), osf=osf.astype(int), rnf=rnf.astype(int)))

def fetch_uci_frame(uci_id, static_name, member=None):
    """UCI 자료 -> DataFrame. ucimlrepo 우선, 실패 시 static zip(경로 패턴 미검증)."""
    try:
        import importlib.util, subprocess, sys
        if importlib.util.find_spec("ucimlrepo") is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ucimlrepo"])
        from ucimlrepo import fetch_ucirepo
        r = fetch_ucirepo(id=uci_id)
        orig = getattr(r.data, "original", None)
        if orig is not None and len(orig):
            return orig
        return pd.concat([r.data.features, r.data.targets], axis=1)
    except Exception as e:
        print(f"ucimlrepo 실패({type(e).__name__}) -> static zip 시도")
        raw = fetch_bytes(f"https://archive.ics.uci.edu/static/public/{uci_id}/{static_name}.zip", timeout=60)
        z = zipfile.ZipFile(io.BytesIO(raw))
        name = member or [n for n in z.namelist() if n.lower().endswith(".csv")][0]
        return pd.read_csv(z.open(name))

def load_ai4i(live=True):
    if live:
        try:
            return parse_ai4i(fetch_uci_frame(601, "ai4i+2020+predictive+maintenance+dataset")), "REAL: UCI AI4I 2020 (합성 생성 자료, CC BY 4.0)"
        except Exception as e:
            print("AI4I 수집 실패 -> 합성 대체:", type(e).__name__, str(e)[:100])
    return synthetic_ai4i(), "SYNTHETIC (NOT real data)"

import zipfile
CMAPSS_COLS = ["unit", "cycle", "op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]

def parse_cmapss_train(text):
    """C-MAPSS train_FDxxx.txt: 공백 구분 26열, 한 행 = 한 엔진의 한 사이클. RUL = 마지막 사이클 - 현재 사이클."""
    df = pd.read_csv(io.StringIO(text) if isinstance(text, str) else io.BytesIO(text), sep=r"\s+", header=None)
    df = df.dropna(axis=1, how="all")
    if df.shape[1] != 26:
        raise ValueError(f"C-MAPSS 열 수 {df.shape[1]} != 26")
    df.columns = CMAPSS_COLS
    df["rul"] = df.groupby("unit")["cycle"].transform("max") - df["cycle"]
    if df["unit"].nunique() < 50:
        raise ValueError("엔진 수 부족")
    return df

def synthetic_cmapss(n_units=100, seed=2026102):
    """NOT real data: 사이클이 진행될수록 일부 센서가 열화하는 합성 엔진 자료."""
    rng = np.random.default_rng(seed); rows = []
    for u in range(1, n_units + 1):
        L = int(rng.integers(130, 360)); c = np.arange(1, L + 1); w = (c / L) ** 2
        r = {"unit": u, "cycle": c, "op1": rng.normal(0, 0.002, L), "op2": rng.normal(0, 0.0003, L), "op3": 100.0}
        for i in range(1, 22):
            base = 100 + 10 * i; sign = 1 if i % 2 else -1
            r[f"s{i}"] = base + (sign * 8 * w if i in (2, 3, 4, 7, 11, 12, 15, 17, 20, 21) else 0) + rng.normal(0, 0.5, L)
        rows.append(pd.DataFrame(r))
    df = pd.concat(rows, ignore_index=True)
    df["rul"] = df.groupby("unit")["cycle"].transform("max") - df["cycle"]
    return df

def load_cmapss_fd001(live=True):
    if live:
        try:
            z = zipfile.ZipFile(io.BytesIO(fetch_bytes("https://data.nasa.gov/docs/legacy/CMAPSSData.zip", timeout=120)))
            return parse_cmapss_train(z.read("train_FD001.txt")), "REAL: NASA C-MAPSS FD001 (시뮬레이션, 라이선스 표기 없음 - 확인 필요)"
        except Exception as e:
            print("C-MAPSS 수집 실패 -> 합성 대체:", type(e).__name__, str(e)[:100])
    return synthetic_cmapss(), "SYNTHETIC (NOT real data)"

In [ ]:
def majority_acc(y):
    return float(y.value_counts(normalize=True).max())

# 1) AI4I
ai4i, KIND_AI4I = load_ai4i(live=True)
print("KIND_AI4I:", KIND_AI4I, "|", ai4i.shape, "| 고장 비율:", round(float(ai4i['machine_failure'].mean()), 4),
      "| 항상 '정상' 예측 정확도:", round(1 - float(ai4i['machine_failure'].mean()), 4), "(정확도는 불균형에서 무의미)")

# 2) Steel Plates
try:
    Xs, ys = parse_steel(fetch_uci_frame(198, "steel+plates+faults")); KIND_STEEL = "REAL: UCI Steel Plates Faults"
except Exception as e:
    print("Steel 수집 실패 -> 합성 대체:", type(e).__name__, str(e)[:100]); Xs, ys = synthetic_steel(); KIND_STEEL = "SYNTHETIC (NOT real data)"
print("KIND_STEEL:", KIND_STEEL, "|", Xs.shape, "| 최빈 클래스 비율:", round(majority_acc(ys), 3))

# 3) SECOM (static zip: 타임스탬프 포함 가능, 경로 패턴 미검증 -> 실패 시 ucimlrepo)
try:
    z = zipfile.ZipFile(io.BytesIO(fetch_bytes("https://archive.ics.uci.edu/static/public/179/secom.zip", timeout=60)))
    Xq, yq = parse_secom(z.read("secom.data").decode(), z.read("secom_labels.data").decode()); KIND_SECOM = "REAL: UCI SECOM (static zip)"
except Exception as e:
    print("SECOM static zip 실패:", type(e).__name__, str(e)[:80])
    try:
        raw = fetch_uci_frame(179, "secom")
        ycol = [c for c in raw.columns if "pass" in str(c).lower() or "fail" in str(c).lower()]
        yq = raw[ycol[0]].map({-1: 0, 1: 1}).rename("fail"); Xq = raw.drop(columns=ycol); KIND_SECOM = "REAL: UCI SECOM (ucimlrepo, 타임스탬프 없음)"
    except Exception as e2:
        print("SECOM ucimlrepo 실패 -> 합성 대체:", type(e2).__name__, str(e2)[:80]); Xq, yq = synthetic_secom(); KIND_SECOM = "SYNTHETIC (NOT real data)"
print("KIND_SECOM:", KIND_SECOM, "|", Xq.shape, "| 불량 비율:", round(float(yq.mean()), 4), "| 결측 비율:", round(float(Xq.isna().mean().mean()), 3))

# 4) C-MAPSS FD001
cm, KIND_CMAPSS = load_cmapss_fd001(live=True)
print("KIND_CMAPSS:", KIND_CMAPSS, "|", cm.shape, "| 엔진 수:", cm['unit'].nunique(), "| 수명(사이클) 중앙값:", int(cm.groupby('unit')['cycle'].max().median()))

# 5) CWRU: 사이트에서 복사한 .mat URL을 넣으면 사용 (미입력/실패 시 합성)
CWRU_URLS = {"normal": "", "outer_race": ""}          # 예: {"normal": "<97.mat 링크>", "outer_race": "<외륜 결함 .mat 링크>"} (확인 필요)
try:
    sig = {k: parse_cwru_mat(fetch_bytes(u, timeout=60)) for k, u in CWRU_URLS.items() if u}
    assert len(sig) == len(CWRU_URLS), "URL 미입력"
    KIND_CWRU = "REAL: CWRU .mat (DE 가속도)"
except Exception as e:
    print("CWRU -> 합성 대체:", type(e).__name__, str(e)[:80])
    sig = {"normal": synthetic_bearing(fault_hz=0.0), "outer_race": synthetic_bearing(fault_hz=105.0)}; KIND_CWRU = "SYNTHETIC (NOT real data)"
print("KIND_CWRU:", KIND_CWRU, {k: {a: round(b, 3) for a, b in bearing_features(v).items()} for k, v in sig.items()})

## 점검 체크리스트 (추가분)
- 표 자료(AI4I, Steel, SECOM)는 **행이 독립**인지, 시간 순서가 의미 있는지(SECOM) 먼저 확인한다. 시간 순서가 있으면 무작위 분할 대신 시간순 분할을 쓴다.
- 클래스 불균형에서는 정확도 대신 PR-AUC와 기준선(항상 정상)을 같이 보고한다.
- CWRU: 같은 녹화에서 자른 세그먼트를 train/test에 섞으면 누수다. 부하·결함 크기 단위로 분할한다(Ch.7).
- C-MAPSS RUL: 사이클 번호는 달력 시간이 아니다. 시뮬레이션 자료의 성능은 실제 설비 성능이 아니다.